# P3-3 · Language model và sinh văn bản

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
import math, torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)
V = 50                                                   # kích thước từ vựng (nhỏ để chạy thử)
tokens = torch.randint(0, V, (2, 8))                     # 2 chuỗi, mỗi chuỗi 8 token
model = nn.Sequential(nn.Embedding(V, 32), nn.Linear(32, V))   # "model" tối giản, chưa train

x = tokens[:, :-1]                     # (B, T)   đầu vào
y = tokens[:, 1:]                      # (B, T)   đáp án: dịch trái 1 vị trí
logits = model(x)                      # (B, T, V)
loss = F.cross_entropy(logits.reshape(-1, V), y.reshape(-1))
print(x.shape, y.shape, logits.shape)
print(f"loss {loss.item():.2f}, model đoán bừa thì khoảng ln V = {math.log(V):.2f}")

torch.Size([2, 7]) torch.Size([2, 7]) torch.Size([2, 7, 50])
loss 3.97, model đoán bừa thì khoảng ln V = 3.91


In [ ]:
@torch.no_grad()
def sample_next(logits, temperature=0.8, top_k=50):
    if temperature == 0:
        return logits.argmax(-1, keepdim=True)                # greedy
    logits = logits / temperature
    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
    logits[logits < v[..., -1, None]] = float("-inf")         # bỏ mọi token ngoài top-k
    return torch.multinomial(torch.softmax(logits, -1), 1)

# thử với 5 token có logit khác nhau: đếm token nào được chọn trong 1.000 lần
torch.manual_seed(0)
logits = torch.tensor([3.0, 2.0, 1.0, 0.0, -1.0])
for t, k in [(0, 50), (0.5, 50), (1.0, 50), (1.0, 2)]:
    picks = torch.cat([sample_next(logits.clone(), t, k) for _ in range(1000)])
    print(f"temperature {t}, top_k {k}:", torch.bincount(picks, minlength=5).tolist())

temperature 0, top_k 50: [1000, 0, 0, 0, 0]
temperature 0.5, top_k 50: [874, 107, 16, 3, 0]
temperature 1.0, top_k 50: [666, 204, 87, 36, 7]
temperature 1.0, top_k 2: [735, 265, 0, 0, 0]


## Bài tập 4: Viết hàm sinh với top-p {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch

def sample_top_p(logits, p=0.9, temperature=1.0):
    probs = torch.softmax(logits / temperature, dim=-1)
    sorted_p, idx = torch.sort(probs, descending=True)
    cum = torch.cumsum(sorted_p, dim=-1)
    keep = cum - sorted_p < p          # giữ token chừng nào tổng trước nó còn < p
    sorted_p = sorted_p * keep
    choice = torch.multinomial(sorted_p / sorted_p.sum(), 1)
    return idx[choice].item()

print(sample_top_p(torch.tensor([2.0, 1.0, 0.0, -1.0]), p=0.9))

1


## Bài tập 5: Model bigram đặt tên người Việt {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
train = ["văn an", "thị lan", "văn minh", "thị hoa", "đức anh", "thị mai", "văn nam", "minh anh", "thu hà",
         "văn hùng", "thị hương", "ngọc lan", "văn long", "thị thu", "minh châu", "văn đức", "thị ngọc", "hải nam"]
test = ["văn hải", "thị hà", "minh long"]
chars = sorted(set("".join(train + test))) + ["."]
idx = {c: i for i, c in enumerate(chars)}
V = len(chars)
# TODO: N (V × V số đếm, đã +1), P (xác suất từng hàng), ppl_test, 5 tên sinh ra

In [ ]:
N = np.ones((V, V))
for w in train:
    s = "." + w + "."
    for a, b in zip(s, s[1:]):
        N[idx[a], idx[b]] += 1
P = N / N.sum(1, keepdims=True)

logp = [np.log(P[idx[a], idx[b]]) for w in test for a, b in zip("." + w + ".", w + ".")]
ppl_test = float(np.exp(-np.mean(logp)))
print(f"perplexity test = {ppl_test:.1f} (đoán đều {V} ký tự thì perplexity = {V})")

rng = np.random.default_rng(0)
for _ in range(5):
    i, out = idx["."], ""
    while True:
        i = rng.choice(V, p=P[i])
        if chars[i] == "." or len(out) > 15:
            break
        out += chars[i]
    print(out)

perplexity test = 8.8 (đoán đều 25 ký tự thì perplexity = 25)
vn  đứâđâứả ơ âh
voo cđđăn

văùocơvouứứmànon
ảiâcịơlị lctơi l


## Bài tập 6: Các cách giải mã trên một model đồ chơi {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
vocab = ["tôi", "thích", "học", "AI", "và", "đọc", "sách", "rất", "nhiều"]
P = np.array([  # hàng: từ hiện tại, cột: từ tiếp theo
    [0, .6, .2, 0, 0, .1, 0, .1, 0], [0, 0, .5, .1, 0, .3, .1, 0, 0], [0, 0, 0, .7, .2, 0, 0, 0, .1],
    [.3, 0, 0, 0, .5, 0, 0, .1, .1], [.6, .1, .1, 0, 0, .2, 0, 0, 0], [0, 0, 0, 0, .1, 0, .8, 0, .1],
    [.2, 0, 0, 0, .5, 0, 0, .2, .1], [0, .7, 0, 0, 0, 0, 0, 0, .3], [.4, 0, 0, 0, .4, 0, 0, .2, 0]])
rng = np.random.default_rng(0)

def loc(p, k=None, top_p=None):
    # TODO: giữ k xác suất lớn nhất, hoặc nhóm nhỏ nhất có tổng ≥ top_p; chuẩn hóa lại
    return p

def sinh(cach, n=12):
    i, out = 0, ["tôi"]
    for _ in range(n - 1):
        p = P[i]
        if cach == "greedy":
            i = int(p.argmax())
        else:
            p = loc(p, k=2) if cach == "top-k" else loc(p, top_p=0.8) if cach == "top-p" else p
            i = rng.choice(len(p), p=p)
        out.append(vocab[i])
    return out

In [ ]:
def loc(p, k=None, top_p=None):
    order = np.argsort(-p)
    if k is not None:
        keep = order[:k]
    else:
        keep = order[: int(np.searchsorted(np.cumsum(p[order]), top_p) + 1)]
    q = np.zeros_like(p); q[keep] = p[keep]
    return q / q.sum()

print("greedy:", " ".join(sinh("greedy")))
tb = {}
for cach in ["thường", "top-k", "top-p"]:
    rng = np.random.default_rng(0)
    tb[cach] = np.mean([len(set(sinh(cach))) for _ in range(200)])
    print(f"{cach:7s}: ví dụ '{' '.join(sinh(cach))}', trung bình {tb[cach]:.1f} từ khác nhau")

greedy: tôi thích học AI và tôi thích học AI và tôi thích
thường : ví dụ 'tôi thích học nhiều và tôi thích đọc sách tôi rất thích', trung bình 6.9 từ khác nhau
top-k  : ví dụ 'tôi thích học và tôi thích học AI và tôi học AI', trung bình 6.1 từ khác nhau
top-p  : ví dụ 'tôi thích học và tôi thích học AI và tôi học AI', trung bình 6.3 từ khác nhau


## Bài tập 7: Tạo nhãn train cho GPT và cho BERT {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch
torch.manual_seed(0)
ids = torch.randint(5, 1000, (40,))       # một chuỗi 40 token
MASK_ID, IGNORE = 0, -100
# TODO: gpt_x, gpt_y; bert_x, bert_y

In [ ]:
gpt_x, gpt_y = ids[:-1], ids[1:]
chon = torch.rand(len(ids)) < 0.15
bert_x = ids.clone(); bert_x[chon] = MASK_ID
bert_y = torch.full_like(ids, IGNORE); bert_y[chon] = ids[chon]
print("GPT  :", gpt_x[:6].tolist(), "→", gpt_y[:6].tolist())
print("BERT :", bert_x[:10].tolist(), "nhãn:", bert_y[:10].tolist())
print(f"GPT học từ {len(gpt_y)} vị trí, BERT chỉ từ {int(chon.sum())} vị trí")

GPT  : [949, 459, 688, 340, 258, 209] → [459, 688, 340, 258, 209, 507]
BERT : [949, 459, 688, 0, 0, 209, 507, 528, 137, 903] nhãn: [-100, -100, -100, 340, 258, -100, -100, -100, -100, -100]
GPT học từ 39 vị trí, BERT chỉ từ 5 vị trí


## Bài tập 8: Ngân sách train theo scaling law {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import math
gpu, flops_moi_gpu, ngay = 8, 3e14, 7
# TODO: C, N, D

In [ ]:
C = gpu * flops_moi_gpu * ngay * 24 * 3600
N = math.sqrt(C / 120)
D = 20 * N
print(f"C = {C:.2e} FLOP → N ≈ {N / 1e9:.2f} tỷ tham số, D ≈ {D / 1e9:.0f} tỷ token")

C = 1.45e+21 FLOP → N ≈ 3.48 tỷ tham số, D ≈ 70 tỷ token
